# v3.4 link E on an A100 — everything at 1 MP, references included

**Arm `VE`** = the v3.4 version (`V34`) with **call 1 on fal's canvas as well**: the reference is generated at ~1 MP (area 1024², aspect kept, up or down, floor 32), exactly as fal makes its own references. Call 2 unchanged from `V34`. This run: the 31-pair failure set at seeds **49/50/51**, so `VE` pairs cell-for-cell with `V34` (link D) and `Vnc` (link C) — the reference resolution is the only new variable. Prediction (RESULTS §6): `g027+p003` recovers the waist-up framing; watch for reference regressions elsewhere.

Open directly: https://colab.research.google.com/github/101011101/magichour_takehome/blob/v3.3-lock/v3/colab/v34_a100.ipynb — Runtime → A100, Run all.


In [ ]:
# 1 · settings
A100_USD_PER_HOUR = 0.689     # CAD/h at 5.3 CU/h x CAD 0.13/CU; edit if your rate differs
SEEDS = [49, 50, 51]          # same as link C, on purpose: paired comparison Vnc vs V34
ARMS = ("VE",)                # link E: fal's canvas on BOTH calls - references at ~1 MP
MATRICES = ["v34_failures.csv"]   # failure set only; add "v34_controls.csv" for the controls
DRIVE_PROJECT_DIR = "Side projects and shi"
PREV_RUN_ZIP = "v33_ironman_run_20260830_0548.zip"   # in Drive v3_runs/: inputs and A4 crops reused

In [ ]:
# 2 · Drive: find the HF cache that holds klein; reuse the iron-man run
import os
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'; BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
KLEIN = 'models--black-forest-labs--FLUX.2-klein-4B'
candidates = [os.path.join(MYDRIVE, 'hf_cache'), os.path.join(BASE, 'tryon_models', 'hf_cache'), os.path.join(BASE, 'hf_cache')]
found = [c for c in candidates if os.path.isdir(os.path.join(c, 'hub', KLEIN))]
os.environ['HF_HOME'] = found[0] if found else candidates[0]
os.environ['V3_MODEL_DIR'] = os.path.join(BASE if os.path.isdir(BASE) else MYDRIVE, 'v3_models')
os.makedirs(os.environ['HF_HOME'], exist_ok=True); os.makedirs(os.environ['V3_MODEL_DIR'], exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'], '(klein cached)' if found else '(no cached klein - cell 4 downloads ~13 GB once)')
PREV = os.path.join(BASE, 'v3_runs', PREV_RUN_ZIP); print('previous run:', PREV, os.path.exists(PREV))

In [ ]:
# 3 · install; pull the bundle from GitHub (public, branch v3.3-lock)
!pip -q install -U diffusers transformers accelerate sentencepiece protobuf mediapipe onnxruntime-gpu opencv-python-headless
!cd /content && rm -rf v34 && wget -q -O v33_ironman_bundle.zip https://github.com/101011101/magichour_takehome/raw/v3.3-lock/v33_ironman_bundle.zip && unzip -qo v33_ironman_bundle.zip -d v34
%cd /content/v34
import os, zipfile, onnxruntime as ort, torch
assert os.path.exists('lib/run_ironman.py') and os.path.exists('v34_failures.csv') and os.path.exists('v34_controls.csv'), 'bundle incomplete'
zipfile.ZipFile(PREV).extractall('run'); print('iron-man inputs + crops unpacked:', len(os.listdir('run/inputs')), 'files')
for f in os.listdir('run/gen'): os.remove('run/gen/' + f)          # a clean gen/ - this run's outputs only
print('onnxruntime providers:', ort.get_available_providers(), '| gpu:', torch.cuda.get_device_name(0))

In [ ]:
# 4 · load klein once, timed
import sys; sys.path.insert(0, 'lib')
import klein_local as K
K.load(); K.info()

In [ ]:
# 5 · one pair first
import run_ironman as R
R.main(MATRICES[0], 'testset', limit=1, seeds=SEEDS[:1], arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
print(sorted(f for f in os.listdir('run/gen')))

In [ ]:
# 6 · both matrices, all seeds (resumable)
for m in MATRICES:
    R.main(m, 'testset', limit=None, seeds=SEEDS, arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
import json; print(json.dumps(json.load(open('run/meta/cost.json')), indent=1))

In [ ]:
# 7 · zip this run's references, outputs and meta to Drive
import shutil, time
name = f"v34_a100_ve_{time.strftime('%Y%m%d_%H%M')}"
with zipfile.ZipFile(f'/content/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('run/refs'):
        if any(a in f for a in ('__VE', '__V34', '__Vnc', '__Vfc')): z.write('run/refs/' + f, 'refs/' + f)
    for f in os.listdir('run/gen'): z.write('run/gen/' + f, 'gen/' + f)
    for f in os.listdir('run/meta'): z.write('run/meta/' + f, 'meta/' + f)
os.makedirs(os.path.join(BASE, 'v3_runs'), exist_ok=True); shutil.copy(f'/content/{name}.zip', os.path.join(BASE, 'v3_runs', name + '.zip'))
print('->', os.path.join(BASE, 'v3_runs', name + '.zip'), ' then locally: python3 v3/build/v34_a100_page.py <unpacked dir> --arm V34')